# Gini Contributions

This notebook demonstrates `gini_contributions` — an instance-level decomposition of the Gini coefficient (Somers' D\_{Y|X}).

Each observation receives a signed contribution:

- A **positive** (label=1) earns credit for each negative it outranks and is penalised for each negative that outranks it.
- A **negative** (label=0) earns credit for each positive that outranks it and is penalised for each positive it outranks.

The contributions are normalised so that `contributions.sum()` equals the overall Gini, making them directly comparable to SHAP-style decompositions.

Topics covered:

- **Single model** contributions and top/bottom contributors
- **Score vs contribution** scatter — where the model's ranking failures are
- **Contributions by score decile** — aggregate lift view
- **Multi-model comparison** — contribution distributions side by side


In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from fastwoe import gini_contributions

%config InlineBackend.figure_format = 'retina'

## 1. Generate Credit Risk Data


In [ ]:
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
n = 10_000

df = pd.DataFrame(
    {
        "age": np.random.randint(18, 75, n),
        "income": np.random.lognormal(10.5, 0.8, n),
        "credit_score": np.random.randint(300, 850, n),
        "debt_to_income": np.random.uniform(0, 1.5, n),
    }
)

default_prob = 1 / (
    1
    + np.exp(
        2.5
        - 0.009 * (df["credit_score"] - 500)
        - 0.1 * np.log(df["income"] / 50_000)
        + 1.5 * df["debt_to_income"]
    )
)
y = np.random.binomial(1, default_prob)

print(f"Dataset: {n:,} samples")
print(f"Default rate: {y.mean():.2%}")

## 2. Train Multiple Models


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.3, random_state=RANDOM_STATE)

models = {
    "Logistic": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=RANDOM_STATE
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, max_depth=5, random_state=RANDOM_STATE
    ),
}

predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict_proba(X_test)[:, 1]
    print(f"{name} trained")

## 3. Single Model Gini Contributions

`gini_contributions` returns:

- `contributions`: array of per-observation contributions, shape `(n,)`
- `gini`: overall Gini coefficient

The contributions sum exactly to the Gini.


In [ ]:
scores = predictions["Gradient Boosting"]
contribs, gini = gini_contributions(scores, y_test)

print(f"Gini coefficient: {gini:.4f}")
print(f"Sum of contributions: {contribs.sum():.4f}")
print(f"Match: {abs(contribs.sum() - gini) < 1e-10}")

In [ ]:
result = pd.DataFrame(
    {
        "score": scores,
        "label": y_test,
        "gini_contribution": contribs,
    },
    index=y_test,
).sort_values("gini_contribution", ascending=False)

print("Top 10 contributors (help Gini most):")
display(result.head(10))

print("Bottom 10 contributors (hurt Gini most):")
display(result.tail(10))

## 4. Score vs Contribution

A scatter of score vs contribution reveals **where the model's ranking failures are**:

- **Positives** at the top of the score range contribute the most (correctly ranked above many negatives).
- **Negatives** at the bottom of the score range also contribute the most (correctly ranked below many positives).
- Observations in the **overlap region** can have negative contributions — these are the misranked pairs.


In [ ]:
colors_map = {1: "#55d3ed", 0: "#69db7c"}

fig, ax = plt.subplots(figsize=(6, 4), dpi=110)

for label_val, label_name in [(1, "Default (1)"), (0, "Non-default (0)")]:
    mask = y_test == label_val
    ax.scatter(
        scores[mask],
        contribs[mask],
        color=colors_map[label_val],
        alpha=0.3,
        s=6,
        label=label_name,
    )

ax.axhline(0, color="black", linestyle="dotted", linewidth=0.8, alpha=0.7)
ax.set_xlabel("Score")
ax.set_ylabel("Gini contribution")
ax.set_title("Score vs Gini Contribution", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, linestyle="dotted", linewidth=0.7, alpha=0.6)
plt.tight_layout()
plt.show()

## 5. Contributions by Score Decile

Aggregating contributions by score decile shows **where in the score distribution the model is gaining or losing Gini**.
A healthy model accumulates most of its Gini in the top deciles.


In [ ]:
decile_df = pd.DataFrame(
    {"score": scores, "label": y_test, "gini_contribution": contribs},
    index=y_test,
)
decile_df["decile"] = pd.qcut(decile_df["score"], q=10, labels=False, duplicates="drop")
decile_df["decile"] = 10 - decile_df["decile"]  # decile 1 = highest scores

agg = (
    decile_df.groupby("decile")
    .agg(
        gini_contribution=("gini_contribution", "sum"),
        default_rate=("label", "mean"),
        n=("label", "count"),
    )
    .reset_index()
)

fig, ax1 = plt.subplots(figsize=(6, 4), dpi=110)
ax2 = ax1.twinx()

bar_colors = ["#55d3ed" if v >= 0 else "#ffa94d" for v in agg["gini_contribution"]]
ax1.bar(
    agg["decile"], agg["gini_contribution"], color=bar_colors, alpha=0.85, label="Gini contribution"
)
ax2.plot(
    agg["decile"],
    agg["default_rate"],
    color="#c430c1",
    marker="o",
    markersize=4,
    label="Default rate",
)

ax1.axhline(0, color="black", linestyle="dotted", linewidth=0.8)
ax1.set_xlabel("Score decile (1 = highest scores)")
ax1.set_ylabel("Gini contribution")
ax2.set_ylabel("Default rate")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax1.set_title("Gini Contribution by Score Decile", fontsize=14)
ax1.set_xticks(agg["decile"])
ax1.grid(True, linestyle="dotted", linewidth=0.7, alpha=0.6)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10, loc="upper right")

plt.tight_layout()
plt.show()

## 6. Multi-Model Comparison

Comparing the contribution distributions across models highlights which model has fewer misranked pairs (narrower negative tail) and where each model accumulates its Gini.


In [ ]:
model_colors = ["#55d3ed", "#69db7c", "#ffa94d"]
model_names = list(predictions.keys())

fig, axes = plt.subplots(1, 3, figsize=(13, 4), dpi=110, sharey=True)

for ax, name, color in zip(axes, model_names, model_colors):
    c, g = gini_contributions(predictions[name], y_test)
    sorted_c = np.sort(c)[::-1]
    cum_c = np.cumsum(sorted_c)

    bar_colors = [color if v >= 0 else "#c430c1" for v in sorted_c]
    ax.bar(range(len(sorted_c)), sorted_c, color=bar_colors, alpha=0.7, width=1.0)
    ax.axhline(0, color="black", linestyle="dotted", linewidth=0.8)
    ax.set_title(f"{name}\nGini: {g:.2%}", fontsize=11)
    ax.set_xlabel("Observations (sorted)")
    ax.grid(True, axis="y", linestyle="dotted", linewidth=0.7, alpha=0.6)
    ax.tick_params(axis="x", labelbottom=False)

axes[0].set_ylabel("Gini contribution")
plt.suptitle("Individual Gini Contributions by Model", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. Cumulative Contribution Curve

Sorting observations by contribution and plotting the cumulative sum shows how quickly each model accumulates its Gini. A steeper initial rise means the model concentrates its discriminatory power in fewer observations.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), dpi=110)

for name, color in zip(model_names, model_colors):
    c, g = gini_contributions(predictions[name], y_test)
    sorted_c = np.sort(c)[::-1]
    cum_c = np.cumsum(sorted_c)
    n_obs = len(cum_c)
    ax.plot(
        np.arange(1, n_obs + 1) / n_obs,
        cum_c,
        color=color,
        label=f"{name} Gini: {g:.2%}",
    )

ax.axhline(0, color="black", linestyle="dotted", linewidth=0.8, alpha=0.7)
ax.set_xlabel("Fraction of observations (sorted by contribution)")
ax.set_ylabel("Cumulative Gini contribution")
ax.set_title("Cumulative Gini Contribution Curve", fontsize=14)
ax.set_xticks(np.arange(0, 1.1, 0.1))
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, linestyle="dotted", linewidth=0.7, alpha=0.6)
plt.tight_layout()
plt.show()